# 🧠 The Perceptron
**Course:** Deep Learning  
**Lecture:** 1 — The Perceptron  
**Part:** b — Perceptron Training

---

> **Where we are:** We know how a neuron computes outputs. Now we will see how it LEARNS weights from data.

**What you'll learn:**
- The Perceptron Learning Rule: $w \leftarrow w + \eta (y - \hat{y}) x$
- Implementing the perceptron from scratch in NumPy
- Convergence on linearly separable vs non-separable data
- Comparing learning rates

**Exam relevance:** ⭐⭐⭐ (The foundation of gradient-based optimization)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

## 📖 Theory: The Perceptron Learning Rule

The perceptron updates its weights only when it makes a mistake.
Update rule:
$$ w \leftarrow w + \eta (y - \hat{y}) x $$
$$ b \leftarrow b + \eta (y - \hat{y}) $$

Where:
- $\eta$ is the learning rate
- $y$ is the true label (0 or 1)
- $\hat{y}$ is the predicted label (0 or 1, output of a step function)

If $y = \hat{y}$, the error is 0, and weights don't change!

## 🔀 Your Options: Learning Rates

At this step, you can choose between learning rates:

| Option | Pros | Cons | When to use |
|---|---|---|---|
| Very small (e.g., 0.001) | Precise updates | Very slow to converge | When data is complex |
| Moderate (e.g., 0.1) | Good balance | Might slightly overshoot | Default choice |
| Large (e.g., 10.0) | Fast initial steps | Oscillates, might never converge | Rarely |

In [ ]:
# ✏️ Your Turn: Implement Perceptron from scratch

class Perceptron:
    def __init__(self, learning_rate=0.1, epochs=10):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None
        self.errors_per_epoch = []
        
    def activation(self, z):
        # Step function: 1 if z >= 0 else 0
        ### YOUR CODE HERE ###
        pass
        
    def predict(self, X):
        ### YOUR CODE HERE ###
        # Compute z and apply activation
        pass
        
    def fit(self, X, y):
        # Initialize weights to zeros
        self.weights = np.zeros(X.shape[1])
        self.bias = 0.0
        
        ### YOUR CODE HERE ###
        # Loop for self.epochs:
        #   Initialize errors = 0
        #   Loop over each sample xi, target yi in zip(X, y):
        #       predict y_hat
        #       update self.weights and self.bias
        #       increment errors if prediction is wrong
        #   append errors to self.errors_per_epoch
        return self

In [ ]:
# ✅ Solution

class PerceptronSol:
    def __init__(self, learning_rate=0.1, epochs=10):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None
        self.errors_per_epoch = []
        
    def activation(self, z):
        return np.where(z >= 0, 1, 0)
        
    def predict(self, X):
        z = np.dot(X, self.weights) + self.bias
        return self.activation(z)
        
    def fit(self, X, y):
        self.weights = np.zeros(X.shape[1])
        self.bias = 0.0
        
        for _ in range(self.epochs):
            errors = 0
            for xi, yi in zip(X, y):
                y_hat = self.predict(xi)
                update = self.lr * (yi - y_hat)
                self.weights += update * xi
                self.bias += update
                errors += int(update != 0.0)
            self.errors_per_epoch.append(errors)
        return self

In [ ]:
# 🔬 Comparison: Learning Rates & Convergence on AND gate

# AND gate dataset
X_and = np.array([[0,0], [0,1], [1,0], [1,1]])
y_and = np.array([0, 0, 0, 1])

# XOR gate dataset
X_xor = np.array([[0,0], [0,1], [1,0], [1,1]])
y_xor = np.array([0, 1, 1, 0])

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
learning_rates = [0.001, 0.1, 10]

for ax, lr in zip(axs, learning_rates):
    p = PerceptronSol(learning_rate=lr, epochs=20)
    p.fit(X_and, y_and)
    ax.plot(range(1, len(p.errors_per_epoch) + 1), p.errors_per_epoch, marker='o')
    ax.set_title(f"AND gate: lr={lr}")
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Number of updates (errors)')

plt.tight_layout()
plt.show()

## 📖 Theory: Convergence on XOR

The perceptron convergence theorem guarantees convergence **if and only if** the classes are linearly separable.
Let's see what happens on the XOR dataset (which is not linearly separable).

In [ ]:
# ✅ Solution: Visualising XOR failure

p_xor = PerceptronSol(learning_rate=0.1, epochs=20)
p_xor.fit(X_xor, y_xor)

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(p_xor.errors_per_epoch) + 1), p_xor.errors_per_epoch, marker='x', color='red')
plt.title('XOR gate: Errors per epoch (Does not converge)')
plt.xlabel('Epochs')
plt.ylabel('Number of updates (errors)')
plt.show()

## 🎓 Exam Corner

### Question 1 (Conceptual)
Write the Perceptron update rule and explain each term.

<details><summary>💡 Answer</summary>

$w \leftarrow w + \eta (y - \hat{y}) x$
- $w$: current weight vector
- $\eta$: learning rate (step size)
- $y$: true label
- $\hat{y}$: predicted label
- $x$: input vector

The term $(y - \hat{y})$ ensures we only update if there's an error. The $x$ term gives the direction to move the weight vector to fix the mistake.
</details>

### Question 2 (Conceptual)
Under what condition is the Perceptron guaranteed to converge?

<details><summary>💡 Answer</summary>

The Perceptron Convergence Theorem states it will only converge (reach zero training error) if the dataset is **linearly separable** (a single hyperplane can perfectly divide the classes).
</details>

### Question 3 (Conceptual)
What is the key limitation of the Perceptron that motivated the MLP?

<details><summary>💡 Answer</summary>

The inability to learn non-linear decision boundaries (like the XOR problem). Multi-Layer Perceptrons (MLPs) overcome this by using hidden layers with non-linear activation functions.
</details>